# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [2]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Detected BASE: /Users/akshatsipany/SST/fde/flasheats-classroom-pack
Database path: /Users/akshatsipany/SST/fde/flasheats-classroom-pack/database/flasheats.db
Database exists: True


## Load the updated FlashEats data

In [3]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?

**Validation contract (assumption → expectation → test → severity):**

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` unique | `len(orders)` vs `nunique(order_id)`; inspect dup rows | HIGH — rate + joins double-count |
| Delivered orders have completion time | delivered ⇒ non-null `actual_delivery_at` | count delivered with null actual | HIGH — denominator undefined |
| Promised ETA is valid | `promised_eta >= created_at`, non-null | violations + nulls | HIGH — delay math invalid |
| Event chronology is valid | `pickup_at <= actual_delivery_at` (delivered) | violation count | MEDIUM — trust in timestamps |
| “Late” has an agreed definition | one owned threshold + population | compute >0 vs >10 vs historical; read metric JSON | HIGH — publishable number changes |

**What could make 56% misleading?** duplicates, `Delivered` vs `delivered` casing, cancelled/null handling, threshold choice (>0 vs >10), and 4 promised<created + 5 actual<pickup rows.

In [ ]:
# Challenge 1 — grain + chronology checks.
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))
print("Duplicate order_id rows:", int(orders["order_id"].duplicated(keep=False).sum()))
if orders["order_id"].duplicated(keep=False).any():
    print("Duplicate business keys (same order_id, conflicting traffic_bucket):")
    display(orders[orders.duplicated("order_id", keep=False)].sort_values("order_id"))

grain = orders.drop_duplicates("order_id", keep="first").copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    grain[c + "_p"] = pd.to_datetime(grain[c], format="mixed", errors="coerce")
grain["final_status_norm"] = grain["final_status"].str.strip().str.lower()
print("Business grain: one row per order_id after dedup →", len(grain), "business orders (raw", len(orders), ").")
print("Delivered (case-insensitive) with missing completion time:",
      int(((grain["final_status_norm"] == "delivered") & grain["actual_delivery_at_p"].isna()).sum()), "(must be UNKNOWN, not on-time)")
print("promised_eta < created_at violations:", int((grain["promised_eta_p"] < grain["created_at_p"]).sum()))
display(grain[grain["promised_eta_p"] < grain["created_at_p"]][["order_id", "created_at", "promised_eta", "final_status"]])
print("pickup_at > actual_delivery_at (delivered) violations:",
      int((((grain["pickup_at_p"] > grain["actual_delivery_at_p"])) & (grain["final_status_norm"] == "delivered")).sum()))
display(grain[(grain["pickup_at_p"] > grain["actual_delivery_at_p"]) & (grain["final_status_norm"] == "delivered")][["order_id", "pickup_at", "actual_delivery_at", "final_status"]])
print("Needs stakeholder clarification: dedup rule, 'Delivered' casing, unknown-delivered handling, whether 9 chronology-violation rows stay in the KPI, and the late threshold owner (no canonical owner documented).")
orders_dedup = grain

## Business Grain:

**One row = one business order (`order_id`) after `drop_duplicates`.** Raw `orders` has 1603 rows but only 1600 unique `order_id`s: `O00120`, `O00723`, `O01302` are duplicated with **conflicting `traffic_bucket`** (e.g. severe vs medium), so any traffic cut or late rate computed on raw rows double-counts. All KPI math below uses the 1600-row deduped grain; `final_status` is normalised case-insensitively (`Delivered` → `delivered`).

In [ ]:
# Correct cross-source grain: tickets (202 rows, 3 null order_id, 1 dup ticket) and restaurant_status are LEFT-joined FROM orders.
# The old cell merged tickets×orders then INNER-joined restaurant_status on order_id — that silently drops all non-ticket, non-status orders.
print("tickets:", tickets.shape, "| restaurant_status:", restaurant_status.shape)
print("Ticket null order_id:", int(tickets["order_id"].isna().sum()), "| dup ticket_id:", int(tickets["ticket_id"].duplicated().sum()))
ticket_orders = tickets.merge(orders_dedup, on="order_id", how="left", indicator="_ticket_map")
print(ticket_orders["_ticket_map"].value_counts(dropna=False))
status_orders = restaurant_status.merge(orders_dedup, on="order_id", how="left", indicator="_status_map", suffixes=("", "_ord"))
print(status_orders["_status_map"].value_counts(dropna=False))
# Kept for later cells; Challenge 4 recomputes coverage properly.
ticket_orders_resturants = ticket_orders.merge(restaurant_status, on="order_id", how="left", suffixes=("", "_rs"))
print("Left-joined ticket×orders shape:", ticket_orders_resturants.shape)

In [ ]:
ticket_orders_resturants.info()
print("Missing actual_delivery_at in the left-joined view:", int(ticket_orders_resturants["actual_delivery_at"].isna().sum()),
      "(includes the 3 null-order tickets + orders with genuinely missing delivery time — NOT proof of a join failure).")

In [ ]:
print("Missing actual_delivery_at (left-joined view):", int(ticket_orders_resturants["actual_delivery_at"].isna().sum()))
print("Of which tickets had NO order_id at all:", int(tickets["order_id"].isna().sum()))
print("Delivered-but-unknown in canonical grain:", int(((orders_dedup["final_status_norm"] == "delivered") & orders_dedup["actual_delivery_at_p"].isna()).sum()))

# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [ ]:
# Challenge 2 — stakeholders disagree on “late”: compute every definition on the SAME grain.
analysis = orders.drop_duplicates("order_id", keep="first").copy()
for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")
analysis["final_status_norm"] = analysis["final_status"].str.strip().str.lower()
analysis["delay_min"] = (analysis["actual_delivery_at"] - analysis["promised_eta"]).dt.total_seconds() / 60
valid_hist = analysis[(analysis["final_status_norm"] == "delivered") & analysis["actual_delivery_at"].notna()].copy()
defs = {
    "VP Ops: any delay > 0 (valid delivered)": float((valid_hist["delay_min"] > 0).mean() * 100),
    "Support Lead: delay > 10 min (valid delivered)": float((valid_hist["delay_min"] > 10).mean() * 100),
    "Finance view: exclude cancelled+unknown (valid delivered)": float((valid_hist["delay_min"] > 0).mean() * 100),
    "Data Team historical: delivered + non-null actual (strict case-sensitive)": float((((analysis["final_status"] == "delivered") & analysis["actual_delivery_at"].notna() & (analysis["delay_min"] > 0)).sum() / max(1, int(((analysis["final_status"] == "delivered") & analysis["actual_delivery_at"].notna()).sum())) * 100)),
    "Naive raw (no dedup/casing/null rules)": float(((pd.to_datetime(orders["actual_delivery_at"], format="mixed", errors="coerce") > pd.to_datetime(orders["promised_eta"], format="mixed", errors="coerce")) & (orders["final_status"] == "delivered") & orders["actual_delivery_at"].notna()).sum() / max(1, int(((orders["final_status"] == "delivered") & orders["actual_delivery_at"].notna()).sum())) * 100),
}
for k, v in defs.items():
    print(f"{k}: {v:.2f}%")
print(f"Denominators: valid delivered={len(valid_hist)}; strict-case delivered={(int(((analysis['final_status']=='delivered') & analysis['actual_delivery_at'].notna()).sum()))}; cancelled={(int((analysis['final_status_norm']=='cancelled').sum()))}; delivered-unknown={(int(((analysis['final_status_norm']=='delivered') & analysis['actual_delivery_at'].isna()).sum()))}")
print("Which to publish? NONE today — the ~56.4% (>0) vs ~23.3% (>10) gap is a business-tolerance choice, and client_metric_definitions.json names NO canonical KPI owner. Owner needed: a single named executive (recommend VP Ops with Finance sign-off) must sign the threshold + population before anything is published.")

# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [ ]:
# Challenge 3 — validate categories WITHOUT silently normalising.
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))
print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))
print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))
print("\n--- Validation verdict ---")
print("final_status: 'Delivered'(5) vs 'delivered' = REPRESENTATION — safe to lower-case/strip once documented; 'cancelled' is SEMANTIC (different population), never merge.")
print("traffic_bucket: 'HIGH'(3) vs 'high' = representation — safe to lower-case; BUT the 3 duplicate order_ids conflict across buckets, so fix grain first.")
print("restaurant status: 'READY'/'Ready'/'ready'/'ready '(trailing space) = representation (safe to strip+lower); 'handoff'(1) vs 'handed_off'(168) = SEMANTIC — needs restaurant-ops owner: does handoff mean courier accepted or restaurant released?; 'unknown'(1) = needs owner (missing instrumentation vs real state?).")
print("support category: 'Late Delivery' vs 'late_delivery' vs 'late_delivery '(trailing space) = representation (safe); 'ETA issue' vs 'eta_changed' vs 'late_delivery' vs 'restaurant_delay' vs 'status_mismatch' vs 'driver_not_moving' vs 'ready_but_waiting' = SEMANTIC — needs Support-lead taxonomy; NEVER collapse without owner.")
print("Action: publish a mapping table with owner sign-off column; block KPI publication on semantic mappings.")

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [ ]:
# Challenge 4 — cross-source integrity (coverage % + risk).
checks = []
checks.append(("orders.restaurant_id → restaurants", float(orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean() * 100)))
checks.append(("orders.driver_id → drivers", float(orders["driver_id"].dropna().isin(drivers["driver_id"]).mean() * 100)))
checks.append(("tickets.order_id → orders (non-null)", float(tickets["order_id"].dropna().isin(orders["order_id"]).mean() * 100)))
checks.append(("restaurant_status.order_id → orders", float(restaurant_status["order_id"].isin(orders["order_id"]).mean() * 100)))
checks.append(("restaurant_status.restaurant_id → restaurants", float(restaurant_status["restaurant_id"].isin(restaurants["restaurant_id"]).mean() * 100)))
mm = restaurant_status.merge(orders_dedup[["order_id", "restaurant_id"]], on="order_id", how="left", suffixes=("_rs", "_ord"))
checks.append(("restaurant_status.restaurant_id == orders.restaurant_id (same order)", float((mm["restaurant_id_rs"] == mm["restaurant_id_ord"]).mean() * 100)))
for name, cov in checks:
    status = "PASS" if cov == 100.0 else ("WARN" if cov >= 99.0 else "FAIL")
    print(f"{name}: {cov:.2f}% [{status}]")
print(f"Tickets with null order_id (unjoinable): {int(tickets['order_id'].isna().sum())} — excluded from coverage denominator, reported separately.")
print("If 1% is unmapped, is that acceptable? Depends on decision: weekly analytics → WARN + disclose; live ETA / restaurant accountability / published KPI → FAIL until mapped or explicitly scoped out. Here ID coverage is 100% (after null handling), so mapping is PASS; the open risks are semantic (category/ETA meaning), not key coverage.")

# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [ ]:
# Challenge 5 — freshness is an SLA question (same record, different verdict per use case).
fresh = restaurant_status.merge(orders_dedup[["order_id", "created_at_p", "promised_eta_p", "pickup_at_p", "actual_delivery_at_p"]], on="order_id", how="left")
fresh["last_updated_at_p"] = pd.to_datetime(fresh["last_updated_at"], format="mixed", errors="coerce")
fresh["hrs_vs_created"] = (fresh["last_updated_at_p"] - fresh["created_at_p"]).dt.total_seconds() / 3600
fresh["after_delivery"] = fresh["last_updated_at_p"] > fresh["actual_delivery_at_p"]
fresh["before_created"] = fresh["last_updated_at_p"] < fresh["created_at_p"]
print(fresh["hrs_vs_created"].describe().to_string())
print("Status updates AFTER actual delivery:", int(fresh["after_delivery"].sum()), "(stale for live use)")
print("Status updates BEFORE order creation:", int(fresh["before_created"].sum()), "(clock/association error)")
display(fresh.sort_values("hrs_vs_created").head(5)[["order_id", "status", "last_updated_at", "created_at_p", "actual_delivery_at_p", "hrs_vs_created"]])
print("Verdict: weekly analytics → PASS (all 502 updates land within ~±35 min of creation, joinable history). Live customer ETA → FAIL (status can lag the delivery it should predict; zero updates arrive after delivery to correct it, and 'unknown'/stale states exist). Restaurant accountability → WARN (usable for trends only after the handoff/handed_off + unknown semantics are owned).")

# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

In [ ]:
validation_report = {
    "business_grain": "WARN — 3 duplicate order_ids with conflicting traffic_bucket; KPI must use 1600-row deduped grain (raw 1603).",
    "timestamp_chronology": "WARN — 4 promised<created + 5 actual<pickup violations; small but KPI-relevant, needs owner disposition.",
    "kpi_definition": "FAIL — >0 min gives ~56.4% but >10 min gives ~23.3%; no canonical owner documented; nothing publishable until threshold+population are signed.",
    "category_semantics": "FAIL — handoff vs handed_off, unknown status, and 9 support-category variants require owner taxonomy; only casing/whitespace is safe to normalise.",
    "cross_source_mapping": "PASS — 100% key coverage after null handling (3 null-order tickets reported separately); no restaurant mismatch.",
    "freshness": "WARN — PASS for weekly analytics, FAIL for live ETA, WARN for accountability.",
    "publish_56_percent": "DO NOT PUBLISH — needs: (1) named KPI owner + signed threshold/population, (2) dedup + casing rules, (3) unknown/chronology disposition, (4) semantic taxonomy sign-off, (5) freshness SLA per use case."
}
for k, v in validation_report.items():
    print(f"{k}: {v}")

# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**